[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/10.%20The%20Dual-Cycling%20Quay%20Crane%20Problem/P10-Tier-1_executed.ipynb)

---


# 10. The Dual-Cycling Quay Crane Problem

## Tier 1 — Markov Decision Process Formulation

### Goal

Represent a small dual-cycling quay crane problem as a Markov decision process (MDP), then use value iteration to select actions that maximize discounted reward while respecting the model's access, crane-position, and stability rules. The notebook also extracts the policy's action sequence, compares it with a simple unload-priority heuristic, and examines sensitivity to the discount factor.

### Inputs and outputs

The example specifies two cranes, import and export containers with locations and operation times, one hatch cover, and reward and time parameters. The computation produces a value function over the states reachable in this model, a greedy action policy, and summaries and plots derived from that policy.

### Model assumptions and scope

- The example is a finite, deterministic MDP. Operation times are represented by fixed action costs; stochastic durations and transition probabilities are not modeled.
- Bays and container locations are discrete. A crane can operate at its current bay or an adjacent bay; an operation places the crane at the target bay. A move action can relocate it by up to two bays.
- Crane interference is modeled by preventing a move to a bay occupied by another crane. The model does not represent travel paths, crossing conflicts, or continuous crane geometry.
- A closed hatch blocks vessel operations at the locations listed for that hatch. Opening the hatch makes those locations accessible.
- Stability is represented by a simplified count imbalance between removed import containers on the two modeled sides. It is a penalty rule, not a vessel stability calculation based on moments or physical loading plans.
- An import container is represented as being carried by a crane after an unload action. A subsequent load action clears that carried status while loading an export container. The terminal condition concerns completion of the listed vessel-side import and export tasks; quayside handoff and any cargo still marked as carried are outside this model.
- The small example is intended to illustrate the MDP formulation and value iteration. Its state-space size and simplified operating rules should not be interpreted as a validated model of a full terminal.

### Solution sequence

1. Define the state information needed to choose future actions.
2. Enumerate feasible actions from each state.
3. Apply the deterministic transition function and calculate the reward.
4. Generate all states reachable from the initial state.
5. Apply value iteration to compute state values and a greedy policy.
6. Follow that policy from the initial state and interpret the resulting measures.

The notebook distinguishes the value function's discounted return from the step-by-step rewards and does not assume that the heuristic comparison must perform worse than the MDP policy.

In [ ]:
import math
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Keep numerical experiments reproducible if random components are added to the example.
np.random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

### Mathematical foundation

An MDP describes sequential decisions. At each step, the system is in a **state** \(s\), a decision-maker chooses an available **action** \(a\), receives a reward, and enters a next state. The state must contain enough information that the legal actions, rewards, and transition law depend on the present state and action, not on the full history.

For this example, write the state as

$$
s = (P, B, U, L, H),
$$

where \(P\) records each crane's position and carrying status, \(B\) records the identity of any carried container, \(U\) is the set of import containers still awaiting an unload action, \(L\) is the set of export containers still awaiting a load action, and \(H\) records hatch-cover states. The implementation also stores a step counter for reporting; it is not part of state identity because no action rule or reward depends on elapsed step count.

For crane \(k\), let \(A_k(s)\) be its feasible actions in state \(s\). The full feasible action set is \(A(s)=\bigcup_k A_k(s)\). An action is represented in the code by a tuple `(crane_id, action_type, target_id)`. The action types used are `unload`, `load`, `move`, and `open_hatch`.

The reward combines action time, a dual-cycle bonus, a stability penalty, and a completion bonus:

$$
R(s,a,s') = -w_{\mathrm{time}}\,t(a)
+ w_{\mathrm{dual}}\,\mathbf{1}_{\mathrm{dual}}(s,a)
- w_{\mathrm{stability}}\,\mathbf{1}_{\mathrm{violation}}(s')
+ 100\,\mathbf{1}_{\mathrm{terminal}}(s').
$$

Here \(t(a)\) is the fixed duration assigned to the action; a dual-cycle indicator is one when a crane carrying an import container performs a load action; a stability indicator is one when the simplified stability rule is violated in the next state; and a terminal indicator is one when no listed import or export tasks remain. The corresponding code is `calculate_reward`.

Because transitions in this example are deterministic, the Bellman optimality equation is

$$
V^*(s) = \max_{a\in A(s)}\left[R(s,a,T(s,a)) + \gamma V^*(T(s,a))\right],
$$

where \(T(s,a)\) is the next state and \(0\leq\gamma<1\) is the discount factor. In a stochastic MDP, the continuation value would instead be an expectation over possible next states; no such randomness is used here.

Value iteration repeatedly applies this Bellman update. With \(V_0(s)=0\), the synchronous update is

$$
V_{n+1}(s) = \max_{a\in A(s)}\left[R(s,a,T(s,a)) + \gamma V_n(T(s,a))\right].
$$

A terminal state has value zero after its entry reward has been received. A nonterminal state with no feasible actions also receives no continuation value in this implementation. The code stops when the maximum change across states is below `theta`, or when `max_iterations` is reached. For a finite state set and a discount factor below one, the Bellman operator is a contraction; iteration therefore converges to its fixed point when continued. If \(S\) is the reachable state set, \(A_{\max}\) is the largest number of feasible actions in any state, and \(I\) is the number of iterations, the value updates take \(O(I|S|A_{\max})\) time. Reachable-state generation additionally evaluates the transitions in the finite reachable graph.

In [ ]:
@dataclass
class Container:
    """Container attributes used by the example MDP."""

    id: str
    type: str
    position: Tuple[int, int]
    weight: float
    unload_time: float = 0.0
    load_time: float = 0.0


@dataclass
class CraneState:
    """A crane's discrete position and carrying status."""

    id: int
    position: Tuple[int, int]
    has_container: bool = False
    container_id: Optional[str] = None


@dataclass
class HatchCover:
    """A hatch cover and the container locations it blocks when closed."""

    id: str
    is_open: bool = False
    blocked_stacks: Optional[List[Tuple[int, int]]] = None

    def __post_init__(self) -> None:
        if self.blocked_stacks is None:
            self.blocked_stacks = []


class MDPState:
    """Complete decision state for the quay-crane example."""

    def __init__(
        self,
        crane_states: List[CraneState],
        unload_containers: List[Container],
        load_containers: List[Container],
        hatch_covers: List[HatchCover],
        time_step: int = 0,
    ) -> None:
        self.crane_states = crane_states
        self.unload_containers = unload_containers
        self.load_containers = load_containers
        self.hatch_covers = hatch_covers
        self.time_step = time_step

    def _key(self) -> tuple:
        """Return the decision-relevant information used for state identity."""
        cranes = tuple(
            sorted(
                (
                    crane.id,
                    crane.position,
                    crane.has_container,
                    crane.container_id,
                )
                for crane in self.crane_states
            )
        )
        unload_ids = tuple(sorted(container.id for container in self.unload_containers))
        load_ids = tuple(sorted(container.id for container in self.load_containers))
        hatches = tuple(
            sorted(
                (
                    hatch.id,
                    hatch.is_open,
                    tuple(sorted(hatch.blocked_stacks or [])),
                )
                for hatch in self.hatch_covers
            )
        )
        return cranes, unload_ids, load_ids, hatches

    def __hash__(self) -> int:
        return hash(self._key())

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, MDPState):
            return NotImplemented
        return self._key() == other._key()

print("MDP state classes are ready.")

In [ ]:
class DualCyclingMDP:
    """Deterministic MDP for the dual-cycling quay-crane example."""

    def __init__(self, num_cranes: int, num_bays: int, containers_per_bay: int):
        self.num_cranes = num_cranes
        self.num_bays = num_bays
        self.containers_per_bay = containers_per_bay

        self.gamma = 0.9
        self.w_dual = 10.0
        self.w_time = 1.0
        self.w_stability = 50.0

        self.base_unload_time = 8.0
        self.base_load_time = 6.0
        self.move_time = 2.0
        self.hatch_open_time = 5.0

        self.initialize_problem()

    def initialize_problem(self) -> None:
        """Create the specified two-crane, seven-container example."""
        self.import_containers = [
            Container("I1", "import", (1, 1), 15.0, self.base_unload_time),
            Container("I2", "import", (1, 2), 18.0, self.base_unload_time + 2),
            Container("I3", "import", (2, 1), 12.0, self.base_unload_time - 2),
            Container("I4", "import", (2, 2), 20.0, self.base_unload_time + 4),
        ]
        self.export_containers = [
            Container("E1", "export", (1, 3), 14.0, load_time=self.base_load_time),
            Container("E2", "export", (2, 3), 16.0, load_time=self.base_load_time + 1),
            Container("E3", "export", (1, 4), 13.0, load_time=self.base_load_time - 1),
        ]

        self.crane_states = [
            CraneState(0, (1, 0)),
            CraneState(1, (2, 0)),
        ]
        self.hatch_covers = [
            HatchCover("H1", False, [(1, 2), (1, 3), (1, 4)])
        ]
        self.initial_state = MDPState(
            crane_states=[
                CraneState(c.id, c.position, c.has_container, c.container_id)
                for c in self.crane_states
            ],
            unload_containers=list(self.import_containers),
            load_containers=list(self.export_containers),
            hatch_covers=[
                HatchCover(h.id, h.is_open, list(h.blocked_stacks or []))
                for h in self.hatch_covers
            ],
        )

    @staticmethod
    def _location_is_accessible(state: MDPState, position: Tuple[int, int]) -> bool:
        """Check hatch access before allowing a vessel-side operation."""
        return all(
            hatch.is_open or position not in (hatch.blocked_stacks or [])
            for hatch in state.hatch_covers
        )

    def get_feasible_actions(self, state: MDPState) -> List[Tuple[int, str, str]]:
        """Enumerate actions satisfying the example's access and position rules."""
        actions: List[Tuple[int, str, str]] = []

        for crane in state.crane_states:
            crane_id = crane.id
            crane_bay = crane.position[0]

            if not crane.has_container:
                for container in state.unload_containers:
                    container_bay = container.position[0]
                    if (
                        abs(crane_bay - container_bay) <= 1
                        and self._location_is_accessible(state, container.position)
                    ):
                        actions.append((crane_id, "unload", container.id))

            if crane.has_container:
                for container in state.load_containers:
                    container_bay = container.position[0]
                    if (
                        abs(crane_bay - container_bay) <= 1
                        and self._location_is_accessible(state, container.position)
                    ):
                        actions.append((crane_id, "load", container.id))

            for target_bay in range(1, self.num_bays + 1):
                if target_bay == crane_bay or abs(target_bay - crane_bay) > 2:
                    continue

                occupied_by_other_crane = any(
                    other.id != crane_id and other.position[0] == target_bay
                    for other in state.crane_states
                )
                if not occupied_by_other_crane:
                    actions.append((crane_id, "move", f"bay_{target_bay}"))

            for hatch in state.hatch_covers:
                if not hatch.is_open and crane_bay == 1:
                    actions.append((crane_id, "open_hatch", hatch.id))

        return actions

    def calculate_reward(
        self,
        state: MDPState,
        action: Tuple[int, str, str],
        next_state: MDPState,
    ) -> float:
        """Combine time cost, dual-cycle reward, stability penalty, and completion reward."""
        crane_id, action_type, target = action

        if action_type == "unload":
            container = next(c for c in state.unload_containers if c.id == target)
            action_time = container.unload_time
        elif action_type == "load":
            container = next(c for c in state.load_containers if c.id == target)
            action_time = container.load_time
        elif action_type == "move":
            action_time = self.move_time
        elif action_type == "open_hatch":
            action_time = self.hatch_open_time
        else:
            raise ValueError(f"Unknown action type: {action_type}")

        reward = -self.w_time * action_time

        if action_type == "load":
            crane = next(c for c in state.crane_states if c.id == crane_id)
            if crane.has_container:
                reward += self.w_dual

        if self.check_stability_violation(next_state):
            reward -= self.w_stability

        if self.is_terminal(next_state):
            reward += 100.0

        return reward

    def check_stability_violation(self, state: MDPState) -> bool:
        """Apply the example's count-imbalance rule to removed import containers."""
        left_side_removed = sum(
            1
            for container in self.import_containers
            if container not in state.unload_containers
            and container.position[0] <= 1
        )
        right_side_removed = sum(
            1
            for container in self.import_containers
            if container not in state.unload_containers
            and container.position[0] > 1
        )
        return abs(left_side_removed - right_side_removed) > 2

    @staticmethod
    def is_terminal(state: MDPState) -> bool:
        """The listed vessel-side tasks are complete when both task sets are empty."""
        return not state.unload_containers and not state.load_containers

    def transition_function(
        self,
        state: MDPState,
        action: Tuple[int, str, str],
    ) -> MDPState:
        """Apply one feasible action and return a separate next-state object."""
        crane_id, action_type, target = action
        new_crane_states = [
            CraneState(c.id, c.position, c.has_container, c.container_id)
            for c in state.crane_states
        ]
        new_unload_containers = list(state.unload_containers)
        new_load_containers = list(state.load_containers)
        new_hatch_covers = [
            HatchCover(h.id, h.is_open, list(h.blocked_stacks or []))
            for h in state.hatch_covers
        ]
        crane = next(c for c in new_crane_states if c.id == crane_id)

        if action_type == "unload":
            container = next(c for c in new_unload_containers if c.id == target)
            new_unload_containers.remove(container)
            crane.has_container = True
            crane.container_id = container.id
            crane.position = (container.position[0], 1)
        elif action_type == "load":
            container = next(c for c in new_load_containers if c.id == target)
            new_load_containers.remove(container)
            crane.has_container = False
            crane.container_id = None
            crane.position = (container.position[0], 1)
        elif action_type == "move":
            target_bay = int(target.split("_")[1])
            crane.position = (target_bay, 0)
        elif action_type == "open_hatch":
            hatch = next(h for h in new_hatch_covers if h.id == target)
            hatch.is_open = True
        else:
            raise ValueError(f"Unknown action type: {action_type}")

        return MDPState(
            new_crane_states,
            new_unload_containers,
            new_load_containers,
            new_hatch_covers,
            state.time_step + 1,
        )

print("Deterministic MDP model is ready.")

In [ ]:
class ValueIteration:
    """Solve the finite discounted MDP with synchronous value iteration."""

    def __init__(
        self,
        mdp: DualCyclingMDP,
        max_iterations: int = 100,
        theta: float = 1e-6,
    ):
        self.mdp = mdp
        self.max_iterations = max_iterations
        self.theta = theta
        self.convergence_history: List[float] = []

    def solve(self) -> Dict:
        """Return the reachable-state values, greedy policy, and convergence data."""
        print("Starting value iteration...")
        all_states = self.generate_reachable_states()
        states = list(all_states)
        print(f"Generated {len(states)} reachable states.")

        values = {state: 0.0 for state in states}
        policy: Dict[MDPState, Tuple[int, str, str]] = {}
        converged = False

        for iteration in range(self.max_iterations):
            previous_values = values
            next_values = dict(previous_values)
            delta = 0.0

            for state in states:
                if self.mdp.is_terminal(state):
                    next_values[state] = 0.0
                    continue

                actions = self.mdp.get_feasible_actions(state)
                if not actions:
                    next_values[state] = 0.0
                    continue

                best_value = float("-inf")
                best_action = None

                for action in actions:
                    next_state = self.mdp.transition_function(state, action)
                    reward = self.mdp.calculate_reward(state, action, next_state)
                    action_value = reward + self.mdp.gamma * previous_values.get(
                        next_state, 0.0
                    )

                    # Retain the first enumerated action when values are numerically tied.
                    if action_value > best_value and not math.isclose(
                        action_value,
                        best_value,
                        rel_tol=1e-12,
                        abs_tol=1e-12,
                    ):
                        best_value = action_value
                        best_action = action

                next_values[state] = best_value
                policy[state] = best_action
                delta = max(delta, abs(previous_values[state] - best_value))

            values = next_values
            self.convergence_history.append(delta)

            if iteration % 10 == 0:
                print(f"Iteration {iteration + 1}: maximum change = {delta:.6g}")

            if delta < self.theta:
                converged = True
                print(f"Converged after {iteration + 1} iterations.")
                break

        if not converged:
            print(
                f"Stopped after {self.max_iterations} iterations; "
                f"final maximum change = {self.convergence_history[-1]:.6g}."
            )

        return {
            "V": values,
            "policy": policy,
            "convergence": self.convergence_history,
            "converged": converged,
        }

    def generate_reachable_states(self) -> Set[MDPState]:
        """Enumerate the complete finite state set reachable from the initial state."""
        reachable: Set[MDPState] = set()
        frontier = deque([self.mdp.initial_state])

        while frontier:
            state = frontier.popleft()
            if state in reachable:
                continue

            reachable.add(state)
            if self.mdp.is_terminal(state):
                continue

            for action in self.mdp.get_feasible_actions(state):
                next_state = self.mdp.transition_function(state, action)
                if next_state not in reachable:
                    frontier.append(next_state)

        return reachable

print("Value iteration solver is ready.")

### Concrete example

The initial state places the two cranes at bays 1 and 2, with four imports awaiting unload actions, three exports awaiting load actions, and hatch H1 closed. The blocked locations for H1 make opening the hatch relevant to both import and export operations at those locations.

The state representation tracks each remaining task and each crane's position and carrying status. An unload action removes an import from the pending set and marks it as carried by the crane. A load action removes an export from the pending set and clears the crane's carrying status. The dual-cycle bonus is awarded only when that load action is performed by a crane marked as carrying an import.

The terminal state is defined by completion of both pending-task sets. This convention and its limits are stated in the scope assumptions above; the terminal condition does not model a separate landside handoff.

In [ ]:
mdp = DualCyclingMDP(num_cranes=2, num_bays=2, containers_per_bay=4)

print("=== MDP problem setup ===")
print("Initial state:")
print(f"  Cranes: {[(c.id, c.position) for c in mdp.initial_state.crane_states]}")
print(
    "  Import containers: "
    f"{[c.id for c in mdp.initial_state.unload_containers]}"
)
print(
    "  Export containers: "
    f"{[c.id for c in mdp.initial_state.load_containers]}"
)
print(
    "  Hatch covers: "
    f"{[(h.id, h.is_open) for h in mdp.initial_state.hatch_covers]}"
)

initial_actions = mdp.get_feasible_actions(mdp.initial_state)
print(f"\nFeasible actions in the initial state: {len(initial_actions)}")
for index, (crane_id, action_type, target) in enumerate(initial_actions[:5], start=1):
    print(f"  {index}. Crane {crane_id}: {action_type} {target}")

if len(initial_actions) > 5:
    print(f"  ... and {len(initial_actions) - 5} additional actions")

In [ ]:
solver = ValueIteration(mdp, max_iterations=50, theta=1e-4)
solution = solver.solve()

V = solution["V"]
policy = solution["policy"]
convergence = solution["convergence"]

print("\n=== Value-iteration results ===")
print(f"Initial-state value: {V[mdp.initial_state]:.4f}")
print(f"Reachable states evaluated: {len(V)}")
print(f"Iterations performed: {len(convergence)}")
print(f"Convergence criterion reached: {solution['converged']}")

In [ ]:
def extract_optimal_sequence(
    mdp: DualCyclingMDP,
    policy: Dict[MDPState, Tuple[int, str, str]],
    max_steps: int = 50,
) -> Tuple[List[Dict], float, int]:
    """Follow the policy and return its trajectory, discounted return, and dual cycles."""
    state = mdp.initial_state
    sequence: List[Dict] = []
    discounted_return = 0.0
    dual_cycles = 0

    for step in range(max_steps):
        if mdp.is_terminal(state):
            print(f"All listed tasks completed in {step} actions.")
            break

        if state not in policy or policy[state] is None:
            raise RuntimeError(f"The policy has no action for the state at step {step}.")

        action = policy[state]
        if action not in mdp.get_feasible_actions(state):
            raise AssertionError("The extracted policy selected an infeasible action.")

        next_state = mdp.transition_function(state, action)
        reward = mdp.calculate_reward(state, action, next_state)
        crane_id, action_type, target = action

        if action_type == "load":
            crane = next(c for c in state.crane_states if c.id == crane_id)
            if crane.has_container:
                dual_cycles += 1

        sequence.append(
            {
                "step": step,
                "action": action,
                "reward": reward,
                "state_info": (
                    f"Pending imports: {len(state.unload_containers)}, "
                    f"pending exports: {len(state.load_containers)}"
                ),
            }
        )
        discounted_return += (mdp.gamma**step) * reward
        state = next_state
    else:
        raise RuntimeError(f"The policy did not reach a terminal state in {max_steps} actions.")

    if not mdp.is_terminal(state):
        raise RuntimeError("Policy extraction ended before all listed tasks were complete.")

    return sequence, discounted_return, dual_cycles


optimal_sequence, total_reward, dual_cycles = extract_optimal_sequence(mdp, policy)

print("\n=== Policy analysis ===")
print(f"Discounted return along the extracted policy: {total_reward:.4f}")
print(f"Dual cycles identified: {dual_cycles}")
print(f"Policy length: {len(optimal_sequence)} actions")
print("\nPolicy actions:")
for step_info in optimal_sequence:
    crane_id, action_type, target = step_info["action"]
    print(
        f"  Step {step_info['step']:2d}: Crane {crane_id} {action_type} {target} "
        f"(Reward: {step_info['reward']:+.1f}; {step_info['state_info']})"
    )

### Heuristic comparison

The comparison policy below is a simple unload-priority heuristic: it selects an available unload action first, then an available load action, and otherwise takes the first feasible action. It is a reference policy, not a second optimization method and not a guaranteed single-cycle strategy.

For a fair comparison of reward, both the heuristic and the value-iteration policy are evaluated using the same discount factor and reward function. The heuristic's number of dual cycles is counted from its actual actions rather than assumed.

In [ ]:
def simulate_unload_priority_heuristic(
    mdp: DualCyclingMDP,
    max_steps: int = 50,
) -> Tuple[float, int, int, bool]:
    """Evaluate an unload-priority heuristic under the MDP's reward definition."""
    state = mdp.initial_state
    discounted_return = 0.0
    steps = 0
    dual_cycles = 0

    while not mdp.is_terminal(state) and steps < max_steps:
        actions = mdp.get_feasible_actions(state)
        if not actions:
            break

        unload_actions = [action for action in actions if action[1] == "unload"]
        load_actions = [action for action in actions if action[1] == "load"]

        if unload_actions:
            action = unload_actions[0]
        elif load_actions:
            action = load_actions[0]
        else:
            action = actions[0]

        crane_id, action_type, _ = action
        if action_type == "load":
            crane = next(c for c in state.crane_states if c.id == crane_id)
            if crane.has_container:
                dual_cycles += 1

        next_state = mdp.transition_function(state, action)
        reward = mdp.calculate_reward(state, action, next_state)
        discounted_return += (mdp.gamma**steps) * reward
        state = next_state
        steps += 1

    return discounted_return, steps, dual_cycles, mdp.is_terminal(state)


heuristic_reward, heuristic_steps, heuristic_dual_cycles, heuristic_completed = (
    simulate_unload_priority_heuristic(mdp)
)

print("\n=== Policy comparison ===")
print(
    f"Value-iteration policy: {total_reward:.4f} discounted return, "
    f"{len(optimal_sequence)} actions, {dual_cycles} dual cycles"
)
print(
    f"Unload-priority heuristic: {heuristic_reward:.4f} discounted return, "
    f"{heuristic_steps} actions, {heuristic_dual_cycles} dual cycles"
)
print(f"Heuristic completed all listed tasks: {heuristic_completed}")
if not math.isclose(heuristic_reward, 0.0, abs_tol=1e-12):
    improvement_percent = (
        (total_reward - heuristic_reward) / abs(heuristic_reward) * 100
    )
    print(
        f"Discounted-return difference: {total_reward - heuristic_reward:.4f} "
        f"({improvement_percent:.2f}% relative to the heuristic's absolute return)"
    )
else:
    print(f"Discounted-return difference: {total_reward - heuristic_reward:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(range(1, len(convergence) + 1), convergence)
axes[0, 0].set_xlabel("Value-iteration sweep")
axes[0, 0].set_ylabel("Maximum value change")
axes[0, 0].set_title("Value-iteration convergence")
axes[0, 0].set_yscale("log")
axes[0, 0].grid(True, alpha=0.3)

methods = ["Value iteration", "Unload-priority heuristic"]
rewards = [total_reward, heuristic_reward]
colors = ["#2ecc71", "#e74c3c"]
bars = axes[0, 1].bar(methods, rewards, color=colors, alpha=0.75)
axes[0, 1].set_ylabel("Discounted return (reward units)")
axes[0, 1].set_title("Policy return comparison")
axes[0, 1].grid(True, alpha=0.3, axis="y")
axes[0, 1].tick_params(axis="x", rotation=12)
for bar, value in zip(bars, rewards):
    offset = 1 if value >= 0 else -1
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset,
        f"{value:.2f}",
        ha="center",
        va="bottom" if value >= 0 else "top",
    )

if optimal_sequence:
    steps = [entry["step"] for entry in optimal_sequence]
    step_rewards = [entry["reward"] for entry in optimal_sequence]
    axes[1, 0].plot(steps, step_rewards, marker="o", linewidth=2, markersize=6)
    axes[1, 0].axhline(y=0, color="black", linestyle="--", alpha=0.4)
axes[1, 0].set_xlabel("Action step")
axes[1, 0].set_ylabel("Immediate reward (reward units)")
axes[1, 0].set_title("Immediate rewards along the extracted policy")
axes[1, 0].grid(True, alpha=0.3)

cycle_labels = ["Value iteration", "Unload-priority heuristic"]
cycle_counts = [dual_cycles, heuristic_dual_cycles]
cycle_bars = axes[1, 1].bar(
    cycle_labels,
    cycle_counts,
    color=["#3498db", "#95a5a6"],
    alpha=0.75,
)
axes[1, 1].set_ylabel("Dual cycles (count)")
axes[1, 1].set_title("Dual cycles in the evaluated policies")
axes[1, 1].grid(True, alpha=0.3, axis="y")
axes[1, 1].tick_params(axis="x", rotation=12)
for bar, value in zip(cycle_bars, cycle_counts):
    axes[1, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        str(value),
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

### What-if analysis: sensitivity to the discount factor

The discount factor γ controls how strongly future rewards contribute to current value. A smaller value places relatively more weight on rewards received soon; a larger value places relatively more weight on later rewards. The analysis below changes only γ, constructs a fresh MDP for each test, and resolves that MDP with the same state and action rules.

The extracted return is discounted using the same γ used by value iteration. Therefore, returns from different tests use different discounting and should be interpreted as results under their respective preferences, not as undiscounted physical performance.

In [ ]:
discount_factors = [0.7, 0.8, 0.9, 0.95]
results = []

for gamma in discount_factors:
    print(f"\nTesting discount factor gamma = {gamma:.2f}")
    mdp_test = DualCyclingMDP(
        num_cranes=2,
        num_bays=2,
        containers_per_bay=4,
    )
    mdp_test.gamma = gamma

    solver_test = ValueIteration(mdp_test, max_iterations=50, theta=1e-4)
    solution_test = solver_test.solve()
    sequence_test, reward_test, dual_cycles_test = extract_optimal_sequence(
        mdp_test,
        solution_test["policy"],
    )

    results.append(
        {
            "gamma": gamma,
            "discounted_return": reward_test,
            "dual_cycles": dual_cycles_test,
            "policy_length": len(sequence_test),
            "converged": solution_test["converged"],
        }
    )

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
gammas = [result["gamma"] for result in results]
returns = [result["discounted_return"] for result in results]
dual_cycles_sensitivity = [result["dual_cycles"] for result in results]
policy_lengths = [result["policy_length"] for result in results]

axes[0].plot(gammas, returns, marker="o", linewidth=2, markersize=8)
axes[0].set_xlabel("Discount factor, gamma")
axes[0].set_ylabel("Discounted return (reward units)")
axes[0].set_title("Discounted return by discount factor")
axes[0].grid(True, alpha=0.3)

axes[1].bar(gammas, dual_cycles_sensitivity, width=0.04, alpha=0.75)
axes[1].set_xlabel("Discount factor, gamma")
axes[1].set_ylabel("Dual cycles (count)")
axes[1].set_title("Dual cycles by discount factor")
axes[1].grid(True, alpha=0.3, axis="y")

axes[2].plot(gammas, policy_lengths, marker="s", linewidth=2, markersize=8)
axes[2].set_xlabel("Discount factor, gamma")
axes[2].set_ylabel("Policy length (actions)")
axes[2].set_title("Policy length by discount factor")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Sensitivity analysis summary ===")
print("Gamma | Discounted return | Dual cycles | Policy actions | Converged")
print("-" * 72)
for result in results:
    print(
        f"{result['gamma']:5.2f} | "
        f"{result['discounted_return']:17.4f} | "
        f"{result['dual_cycles']:11d} | "
        f"{result['policy_length']:14d} | "
        f"{result['converged']}"
    )

### Results interpretation

The initial-state value is the model's estimate of the best achievable discounted return from the initial state. The extracted policy provides a concrete sequence of actions; its immediate rewards can be discounted and summed to compare the trajectory with the value estimate. Small differences can occur because value iteration stops at a numerical tolerance rather than continuing indefinitely.

The dual-cycle count measures load actions performed while the acting crane is marked as carrying an import. It is a model-defined operational indicator, not a measure of crane productivity per hour. The heuristic comparison shows how one specified rule performs on this instance; it does not establish that every heuristic performs worse than value iteration.

The sensitivity analysis shows how changing the discount factor affects the policy and its discounted return. A change in return partly reflects the changed weighting of future rewards itself, so the return values at different discount factors are not directly comparable as undiscounted totals.

The stability rule is intentionally simplified. In this particular instance, the difference in removed-container counts between the two modeled sides cannot exceed the rule's threshold of two, so this constraint does not generate a stability penalty. Hatch access and endpoint crane interference are represented in the feasible-action rules; continuous travel conflicts and physical vessel stability are outside the model.

In [ ]:
assert mdp.initial_state in V
assert mdp.is_terminal(
    mdp.transition_function(
        optimal_sequence[-1]["action"] and mdp.initial_state,
        optimal_sequence[-1]["action"],
    )
) is False if len(optimal_sequence) == 1 else True
assert len(optimal_sequence) > 0
assert all(
    entry["action"] is not None
    and math.isfinite(entry["reward"])
    for entry in optimal_sequence
)

print("\n" + "=" * 64)
print("DUAL-CYCLING QUAY CRANE PROBLEM — TIER 1 SUMMARY")
print("=" * 64)
print(
    f"Problem instance: {mdp.num_cranes} cranes, "
    f"{len(mdp.import_containers)} imports, "
    f"{len(mdp.export_containers)} exports"
)
print(f"Reachable states evaluated: {len(V)}")
print(
    f"Value-iteration sweeps: {len(convergence)} "
    f"(convergence criterion reached: {solution['converged']})"
)
print("\nVALUE-ITERATION POLICY")
print(f"  Initial-state value: {V[mdp.initial_state]:.4f}")
print(f"  Extracted discounted return: {total_reward:.4f}")
print(f"  Dual cycles: {dual_cycles}")
print(f"  Action count: {len(optimal_sequence)}")
print("\nUNLOAD-PRIORITY HEURISTIC")
print(f"  Discounted return: {heuristic_reward:.4f}")
print(f"  Dual cycles: {heuristic_dual_cycles}")
print(f"  Completed listed tasks: {heuristic_completed}")
print("\nModel checks")
print("  Extracted policy actions are feasible in their visited states.")
print("  Extracted policy reaches the model's terminal condition.")
print("  Rewards and returns are computed from the stated transition model.")
print("=" * 64)